# Cum am construit `app/app.py` — tutorial Gradio (pas cu pas)

Ideea Gradio, într-o propoziție: **o funcție Python devine o interfață web.**
Tu scrii funcția, Gradio face caseta, butonul și layout-ul.

Construim app-ul incremental, exact în ordinea în care e scris `app.py`:
funcția simplă -> mai multe input-uri -> tab Chat -> starea partajată ->
regula subiect/știre -> tab Agent -> punem tab-urile împreună -> recapitulare.

Inspirat din [Gradio Quickstart](https://www.gradio.app/guides/quickstart).
Regula tutorialului: **cât mai simplu, doar esențialul.**

> `app/app.py` este doar un strat subțire de Gradio peste `core/` (agent, graph),
> construit în cursurile C2–C7. Aici nu rescriem `core/` — îl chemăm.
> Ca să ruleze fără chei API, folosim un backend fals.

In [1]:
%pip install -q gradio
import gradio as gr
print("Gradio", gr.__version__)


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\valer\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\valer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio 6.14.0


## 1. Cel mai simplu Gradio

`gr.Interface` are nevoie de 3 lucruri: `fn` (funcția), `inputs`, `outputs`.

In [2]:
def saluta(nume):
    return "Salut, " + nume

gr.Interface(fn=saluta, inputs="text", outputs="text").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Atât. Caseta, butonul *Submit*, totul l-a făcut Gradio. Pe asta se construiește
toată aplicația.

## 2. Mai multe input-uri = o listă

Tab-urile noastre au mai multe câmpuri. Dacă funcția are mai multe argumente,
dai o **listă** la `inputs` (ordinea = ordinea argumentelor).

In [ ]:
def combina(text, optiune, numar):
    return f"[{optiune} @ {numar}] {text}"

gr.Interface(
    fn=combina,
    inputs=[gr.Textbox(label="Text"),
            gr.Dropdown(["a", "b"], value="a", label="Opțiune"),
            gr.Slider(0, 1, value=0.3, step=0.1, label="Număr")],
    outputs=gr.Textbox(label="Rezultat"),
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Reține tiparul `[text, dropdown, slider] -> funcție -> text`. **Asta e un tab.**

## 3. Backend fals (ca să rulăm fără chei API)

În `app.py` real, sus, sunt 2 importuri din `core/` (construite în C6–C7):

```python
from core.agent import generate_agent_response   # un agent RAG
from core.graph import run_thread                 # dezbatere multi-agent
```

Aici le înlocuim cu funcții-jucărie. Restul codului rămâne identic ca structură.

In [4]:
def fake_llm(prompt):
    return "(răspuns simulat) despre: " + prompt[:70]

def fake_agent(slug, stimulus):
    voci = {"anti_sistem": "Instituțiile par din nou rupte de oameni.",
            "pro_european": "Să discutăm pe baza procedurilor."}
    return voci.get(slug, f"[{slug}] {stimulus[:50]}")

AGENTS = [("Anti-sistem", "anti_sistem"), ("Pro-european", "pro_european")]
print("backend fals pregătit")

backend fals pregătit


## 4. Primul tab real: Chat

În `app.py`, tab-ul Chat e funcția `chat()` + un `gr.Interface`. O reproducem
cu `fake_llm`.

In [5]:
def chat(prompt):
    return fake_llm(prompt) if prompt.strip() else "Scrie un prompt."

gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Întrebare / prompt", lines=4),
    outputs=gr.Textbox(label="Răspuns", lines=10),
    title="Chat",
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Tab-ul Chat complet, fără să scriem vreun buton. Gradio l-a făcut.

## 5. Starea partajată: `CFG` și `ART`

Tab-ul **Setări** alege modelul și (opțional) încarcă o știre. Celelalte tab-uri
trebuie să **vadă** acea știre. Soluția minimă: două dicționare la nivel de modul.

- `CFG` = provider / model / temperatură
- `ART` = textul + titlul știrii încărcate

Setări **scrie** în ele, restul tab-urilor **citesc**. (Alternativa „canonică"
ar fi `gr.State`; varianta minimă alege simplitatea.)

Truc din `app.py`: provider + model sunt **un singur dropdown**
(`"provider|model"`) — imposibil să fie nepotrivite.

In [6]:
CFG = {"provider": "gemini", "model": "gemini-2.5-flash-lite", "temp": 0.3}
ART = {"text": "", "title": ""}

MODEL_CHOICES = [("gemini · gemini-2.5-flash-lite", "gemini|gemini-2.5-flash-lite"),
                 ("deepseek · deepseek-chat",       "deepseek|deepseek-chat")]

def setup(model_choice, temperature, fake_url):
    provider, model = model_choice.split("|", 1)     # despărțim "provider|model"
    CFG.update(provider=provider, model=model, temp=temperature)
    if fake_url.strip():
        ART.update(text=f"Text fals al știrii de la {fake_url}", title=fake_url)
        return f"Setări salvate. Știre ACTIVĂ: {fake_url}"
    ART.update(text="", title="")
    return f"Setări salvate ({provider} · {model}). Fără știre."

gr.Interface(
    fn=setup,
    inputs=[gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1],
                        label="Provider · Model"),
            gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
            gr.Textbox(label="URL știre (gol = fără știre)")],
    outputs=gr.Textbox(label="Stare"),
    title="Setări",
).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 6. Regula cheie: subiectul intră *peste* știre

`_subject()` decide ce primește un agent:

- **știre + subiect** -> vorbim despre subiect, dar **în contextul știrii**
- **doar știre** -> vorbim despre știre
- **doar subiect** -> vorbim doar despre subiect

In [7]:
def _subject(typed):
    typed = (typed or "").strip()
    news = ART["text"].strip()
    if news and typed:
        return f"{typed}\n\n[În contextul acestei știri:]\n{news[:600]}"
    if news:
        return news[:700]
    return typed

ART.update(text="Știre despre UE și energie.")
print(_subject("Bolojan"))     # subiect peste știre
ART.update(text="")
print(_subject("Bolojan"))     # doar subiect

Bolojan

[În contextul acestei știri:]
Știre despre UE și energie.
Bolojan


## 7. Tab-ul Agent

Tab-ul Agent = `_subject()` + chemarea backend-ului. În `app.py` real,
`fake_agent` e `generate_agent_response` din `core.agent` (C6).

In [8]:
def agent(text, slug):
    s = _subject(text)
    if not s.strip():
        return "Încarcă o știre sau scrie un subiect."
    return fake_agent(slug, s)               # în app: generate_agent_response(...)

gr.Interface(
    fn=agent,
    inputs=[gr.Textbox(label="Subiect (intră peste știre, dacă e încărcată)",
                       lines=3),
            gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    outputs=gr.Textbox(label="Comentariu", lines=10),
    title="Agent",
).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Tab-urile **Rezumat**, **Toți agenții** și **Dezbatere** au exact același tipar:
o funcție + un `gr.Interface`. Doar funcția diferă (rezumat / loop pe roluri /
`core.graph.run_thread`).

## 8. Punem tab-urile împreună

`app.py` are 6 tab-uri cu o **temă comună**. `gr.TabbedInterface` nu acceptă
`theme=` pe toate versiunile, așa că facem ce face el intern: un `gr.Blocks`
cu temă, `gr.Tabs`, și randăm fiecare `Interface` cu `.render()`.

In [9]:
tab_setup = gr.Interface(setup,
    [gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1], label="Provider · Model"),
     gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
     gr.Textbox(label="URL știre")],
    gr.Textbox(label="Stare"), title="Setări")

tab_chat = gr.Interface(chat, gr.Textbox(label="Prompt", lines=3),
    gr.Textbox(label="Răspuns", lines=8), title="Chat")

tab_agent = gr.Interface(agent,
    [gr.Textbox(label="Subiect", lines=3),
     gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    gr.Textbox(label="Comentariu", lines=8), title="Agent")

TABS = [("Setări", tab_setup), ("Chat", tab_chat), ("Agent", tab_agent)]

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown("# EchoChamber Studio")
    with gr.Tabs():
        for nume, iface in TABS:
            with gr.Tab(nume):
                iface.render()

demo.launch()

C:\Users\valer\AppData\Local\Temp\ipykernel_28568\3885253093.py:17: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Acesta e scheletul exact din `app.py`. În aplicația reală sunt 6 tab-uri în
loc de 3, iar funcțiile cheamă `core/` în loc de `fake_*`.

## 9. Recapitulare

**Ce face Gradio (tot tutorialul, pe scurt):**

1. `gr.Interface(fn, inputs, outputs)` — o funcție devine interfață.
2. Mai multe input-uri = o listă. Un astfel de bloc = un tab.
3. `CFG` / `ART` (dict-uri de modul) = starea partajată: Setări scrie, restul citesc.
4. `_subject()` = regula subiect-peste-știre.
5. `gr.Blocks` + `gr.Tabs` + `.render()` = cele 6 tab-uri cu temă comună.

**Dependențe (din structura repo):** `app/app.py` cheamă doar `core/` —
nu rescrie nimic.

| Tab(uri) | Funcție în app.py | Backend | Curs |
|---|---|---|---|
| Setări · Chat · Rezumat | `setup` · `chat` · `summary` | apel LLM direct | C2 |
| Agent | `agent` -> `_agent` | `core.agent` (FAISS + rol) | C5 + C6 |
| Toți agenții | `all_agents` | loop pe `roles.yaml` -> `core.agent` | C6 |
| Dezbatere | `debate` | `core.graph.run_thread` (LangGraph) | C7 |

`core.agent` -> `core.retriever` (FAISS) + `roles.yaml` + LLM.
`core.graph` orchestrează `core.agent` (round-robin). Singura punte offline->runtime:
vectorstore-urile construite offline, citite de retriever la fiecare cerere.

**Mesajul cheie:** aplicația nu e un proiect nou. E un strat subțire Gradio
peste funcțiile din C2–C7. Fiecare tab = un buton peste o funcție de curs.

# TODO - Tema 3

## TODO final — modifică aplicația Gradio din notebook

În ultima parte a notebook-ului, continuă de la aplicația Gradio construită mai sus și adaugă o mică extensie individuală.

Nu trebuie să refaci aplicația de la zero. Modifică direct codul existent din notebook.

Trebuie să adaugi cel puțin **3 modificări vizibile**:

1. **Un tab nou**

   Exemplu: `Despre`, `Etică`, `Ajutor`, `Rezumat`, `Export`, `Analiză`.

   Tabul trebuie să apară în aplicație când rulezi ultima celulă.

2. **O funcție simplă nouă**

   Exemplu:

   - rezumă textul introdus;

   - numără cuvintele;

   - curăță textul;

   - transformă răspunsul într-o variantă mai scurtă;

   - formatează rezultatul pentru copiere.

3. **Un element de design**

   Exemplu:

   - titlu mai bun;

   - subtitlu;

   - emoji pentru taburi;

   - altă temă Gradio;

   - card vizual pentru rezultat;

   - layout mai clar cu `gr.Row()` sau `gr.Column()`.

Opțional, poți adăuga și:

4. **O opțiune de utilizator**

   Exemplu:

   - dropdown pentru tipul de răspuns;

   - slider pentru lungimea răspunsului;

   - selector pentru ton;

   - checkbox pentru răspuns scurt/lung.

   

### Cerință minimă

La final, aplicația trebuie să ruleze în notebook și modificările trebuie să fie vizibile în interfață.

### Scrie sub cod, în 3–4 propoziții:

- Ce am adăugat:

- Ce funcție nouă am creat:

- Ce element de design am modificat:

- Ce aș îmbunătăți dacă aș continua aplicația:



In [11]:
import gradio as gr
from datetime import datetime
import re

# === FUNCȚIE NOUĂ 1: detector de markeri discursivi ===
# specifică pentru EchoChamber: caută cuvinte-cheie tipice fiecărei bule
MARKERI_BULE = {
    "anti_sistem": ["sistem", "mafia", "corupți", "hoți", "minciună", "manipulare",
                    "BOR", "elite", "marionete", "vânduți", "trădare"],
    "pro_european": ["procedură", "instituții", "stat de drept", "UE", "valori",
                     "democrație", "transparență", "reformă", "dialog", "consens"],
}

def detector_markeri(text, bula):
    """Detectează cât de 'încărcat' retoric e un text pentru o anumită bulă."""
    if not text or not text.strip():
        return "Lipsește textul de analizat."

    text_lower = text.lower()
    cuvinte_totale = len(text.split())
    if cuvinte_totale == 0:
        return "Text gol."

    markeri = MARKERI_BULE.get(bula, [])
    gasite = [m for m in markeri if m.lower() in text_lower]
    nr_markeri = len(gasite)

    # scor: % din cuvintele totale care sunt markeri
    scor = round(100 * nr_markeri / max(cuvinte_totale, 1), 2)

    if scor >= 8:
        verdict = "🔴 Intensitate mare — textul folosește vocabular puternic marcat."
    elif scor >= 3:
        verdict = "🟡 Intensitate medie — câteva semnale clare."
    else:
        verdict = "🟢 Intensitate mică — text aproape neutru."

    return f"""🎯 Detector markeri discursivi — bula `{bula}`

Cuvinte totale:       {cuvinte_totale}
Markeri găsiți:       {nr_markeri}  ({", ".join(gasite) if gasite else "—"})
Scor intensitate:     {scor}%

{verdict}
"""

tab_markeri = gr.Interface(
    detector_markeri,
    [gr.Textbox(label="Text de analizat", lines=6,
                placeholder="Lipește un comentariu sau un răspuns de agent..."),
     gr.Dropdown(list(MARKERI_BULE.keys()), value="anti_sistem", label="Profil discursiv")],
    gr.Textbox(label="Analiză", lines=10),
    title="🎯 Markeri discursivi"
)


# === FUNCȚIE NOUĂ 2: comparație între două texte ===
def compara_texte(text_a, text_b, bula):
    """Compară două texte după densitatea markerilor unei bule."""
    if not text_a.strip() or not text_b.strip():
        return "Am nevoie de ambele texte ca să compar."

    markeri = MARKERI_BULE.get(bula, [])

    def scor(t):
        cuvinte = max(len(t.split()), 1)
        gasite = sum(1 for m in markeri if m.lower() in t.lower())
        return round(100 * gasite / cuvinte, 2)

    s_a, s_b = scor(text_a), scor(text_b)
    if s_a > s_b:
        verdict = f"Textul A e mai aproape de profilul `{bula}` (+{round(s_a - s_b, 2)}%)."
    elif s_b > s_a:
        verdict = f"Textul B e mai aproape de profilul `{bula}` (+{round(s_b - s_a, 2)}%)."
    else:
        verdict = "Egalitate — ambele texte au aceeași densitate de markeri."

    return f"Scor A: {s_a}%  ·  Scor B: {s_b}%\n\n{verdict}"

tab_compara = gr.Interface(
    compara_texte,
    [gr.Textbox(label="Textul A", lines=4),
     gr.Textbox(label="Textul B", lines=4),
     gr.Dropdown(list(MARKERI_BULE.keys()), value="anti_sistem", label="Profil discursiv")],
    gr.Textbox(label="Rezultat comparație", lines=6),
    title="⚖️ Comparație bule"
)


# === TAB DESPRE (reformulat puțin) ===
def info_aplicatie(_):
    return """
### 🗣️ EchoChamber Studio

Simulator de bule discursive românești, construit peste un corpus de comentarii YouTube
și mai mulți agenți cu roluri distincte.

**Cum funcționează pe scurt:**
- fiecare agent are un rol (`roles.yaml`) și un mini-corpus propriu (50 comentarii, C5);
- răspunsurile vin de la un LLM (Gemini / DeepSeek) cu context recuperat din FAISS;
- tab-ul Dezbatere orchestrează mai mulți agenți printr-un graf LangGraph (C7).

**Limite și etică:**
- agenții *nu* sunt persoane reale, sunt caricaturi discursive;
- corpusul e selectat, deci e părtinitor prin construcție;
- output-ul nu e sondaj de opinie și nu trebuie tratat ca atare;
- orice utilizare publică necesită moderare umană.

**Bula pe care am lucrat:** `anti_sistem`.
"""

tab_despre = gr.Interface(
    info_aplicatie,
    gr.Textbox(visible=False, value="trigger"),
    gr.Markdown(),
    title="ℹ️ Despre",
    submit_btn="Afișează"
)


# === TAB-URILE EXISTENTE, cu emoji ===
tab_setup_v2 = gr.Interface(setup,
    [gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1], label="Provider · Model"),
     gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
     gr.Textbox(label="URL știre")],
    gr.Textbox(label="Stare"), title="⚙️ Setări")

tab_chat_v2 = gr.Interface(chat,
    gr.Textbox(label="Prompt", lines=3),
    gr.Textbox(label="Răspuns", lines=8), title="💬 Chat")

tab_agent_v2 = gr.Interface(agent,
    [gr.Textbox(label="Subiect", lines=3),
     gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    gr.Textbox(label="Comentariu", lines=8), title="🎭 Agent")


# === DESIGN SCHIMBAT: temă diferită + header cu Row/Column ===
TABS_V2 = [
    ("Setări",     tab_setup_v2),
    ("Chat",       tab_chat_v2),
    ("Agent",      tab_agent_v2),
    ("Markeri",    tab_markeri),    # tab nou 1
    ("Comparație", tab_compara),    # tab nou 2
    ("Despre",     tab_despre),
]

azi = datetime.now().strftime("%d.%m.%Y · %H:%M")

with gr.Blocks(theme=gr.themes.Citrus()) as demo_v2:
    with gr.Row():
        with gr.Column(scale=3):
            gr.Markdown("# 🗣️ EchoChamber Studio")
            gr.Markdown("*Simulator de bule discursive românești — extensie student*")
        with gr.Column(scale=1):
            gr.Markdown(f"**Rulare:** {azi}\n\n**Bulă activă:** `anti_sistem`")

    gr.Markdown("> ⚠️ Toate răspunsurile sunt generate de AI pe baza unui corpus selectat. Nu reprezintă opinii reale.")

    with gr.Tabs():
        for nume, iface in TABS_V2:
            with gr.Tab(nume):
                iface.render()

demo_v2.launch()

C:\Users\valer\AppData\Local\Temp\ipykernel_28568\1729874374.py:150: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Citrus()) as demo_v2:


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


- **Ce am adăugat:** Două tab-uri noi — „Markeri" (analizează un text) și „Comparație" (pune două texte față-n față) — plus un tab „Despre" cu context și limite etice ale aplicației.

- **Ce funcție nouă am creat:** `detector_markeri(text, bula)` care numără câte cuvinte-cheie specifice unei bule discursive (ex: „sistem", „mafia", „corupți" pentru `anti_sistem`) apar într-un text și calculează un scor de intensitate retorică (% din total cuvinte), cu verdict pe trei niveluri (mic / mediu / mare).

- **Ce element de design am modificat:** Am schimbat tema pe `gr.themes.Citrus()`, am adăugat emoji la fiecare tab, și am refăcut header-ul cu `gr.Row()` + două `gr.Column()` — titlul și subtitlul în stânga, data rulării și bula activă în dreapta, plus un disclaimer vizibil sub header.

- **Ce aș îmbunătăți dacă aș continua aplicația:** Lista de markeri e acum hardcodată după intuiție — aș extrage-o automat din corpusul fiecărei bule cu TF-IDF pe comentariile din C5, ca să fie obiectivă. În plus, aș conecta tab-ul „Markeri" direct la output-ul tab-ului „Agent", ca să poți inspecta automat ce generează modelul fără copy-paste.